# Prediction demo
Fits ridge regression on (1) raw location embeddings and (2) SPLICE sparse codes,
then evaluates both and inspects top concepts.

In [1]:
import sys
from pathlib import Path
import argparse

import numpy as np
import torch

sys.path.append(str(Path(".").resolve().parent))

## 1. Config

In [ ]:
TASK        = "pop_density"          # any key from TASK_CONFIGS
DATA_ROOT   = "~/data/prediction_tasks"
MODEL_PATH  = "/home/libe2152/outputs/explainable-earth-embeddings/modified-git-10M/txt-open_clip__loc-satclip__tproj-linear__lproj-none__tft-lora__lft-only_proj__lora_r-4__loss-clip__lr-0.0001__sub-100000/best.pt"                   # path to location-text checkpoint (.pt)
SPLICE_PATH = "/home/libe2152/projects/explainable-earth-embeddings/splice_results/splice_model.pt"                   # path to splice_model.pt
CONCEPTS_PT = "/home/libe2152/projects/explainable-earth-embeddings/splice_results/concepts.pt"                       # path to concepts.pt
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")

device: cuda


## 2. Load dataset

In [3]:
from prediction.dataset import GeoDataset

# for now subsample, in general use all but not fast
train_ds = GeoDataset(TASK, root=DATA_ROOT, split="train", subsample=1000)
test_ds  = GeoDataset(TASK, root=DATA_ROOT, split="test", subsample= 200)

print(f"train: {len(train_ds)}  test: {len(test_ds)}")

y_train = train_ds.values
y_test  = test_ds.values

train: 1000  test: 200


## 3. Load location model and SPLICE

In [4]:
sys.path.append("../..")

In [5]:
from splice import _LocWrapper

In [6]:
from models.model import build_model

def load_location_model(model_path, device):
    ckpt = torch.load(model_path, map_location=device)
    args = argparse.Namespace(**ckpt["args"])
    args.precomputed_location_embeddings = False
    model = build_model(
        text_encoder=args.text_encoder,
        location_encoder=args.location_encoder,
        text_projection=args.text_projection,
        location_projection=args.location_projection,
        shared_dim=args.shared_dim,
        text_finetune_mode=args.text_finetune_mode,
        loc_finetune_mode=args.loc_finetune_mode,
        text_proj_hidden_layers=args.text_proj_hidden_layers,
        text_proj_hidden_features=args.text_proj_hidden_features,
        loc_proj_hidden_layers=args.loc_proj_hidden_layers,
        loc_proj_hidden_features=args.loc_proj_hidden_features,
        text_nonlinearity=args.text_nonlinearity,
        loc_nonlinearity=args.loc_nonlinearity,
        precomputed_text_embeddings=args.precomputed_text_embeddings,
        precomputed_location_embeddings=args.precomputed_location_embeddings,
        device=device,
    )
    model.load_state_dict(ckpt["model"], strict=False)
    model.eval()
    return model.location_encoder

location_model = load_location_model(MODEL_PATH, DEVICE)
splice_model   = torch.load(SPLICE_PATH, map_location=DEVICE, weights_only=False)
splice_model.eval()
print("models loaded")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...23}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.embeddings.class_embedding                        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.l

using pretrained moco vit16
models loaded


## 4. Ridge on raw embeddings

In [7]:
from prediction.sparse_codes import get_embeddings
from prediction.model import fit_ridge
from prediction.eval import eval_ridge

emb_train = get_embeddings(train_ds.latlons, location_model, device=DEVICE)
emb_test  = get_embeddings(test_ds.latlons,  location_model, device=DEVICE)

probe_emb  = fit_ridge(TASK, emb_train, y_train)
score_emb  = eval_ridge(TASK, probe_emb, emb_test, y_test)
print(f"Raw embeddings  — score: {score_emb:.4f}")

[treecover] best alpha: 1
Raw embeddings  — score: 0.4196


## 5. Ridge on SPLICE sparse codes

In [8]:
from prediction.sparse_codes import get_sparse_codes

codes_train = get_sparse_codes(train_ds.latlons, location_model, splice_model, device=DEVICE)
codes_test  = get_sparse_codes(test_ds.latlons,  location_model, splice_model, device=DEVICE)

probe_codes = fit_ridge(TASK, codes_train, y_train)
score_codes = eval_ridge(TASK, probe_codes, codes_test, y_test)
print(f"Sparse codes    — score: {score_codes:.4f}")
print(f"Delta (codes - emb): {score_codes - score_emb:+.4f}")

[treecover] best alpha: 0.1
Sparse codes    — score: 0.4424
Delta (codes - emb): +0.0228


## 6. Top concepts

In [9]:
from prediction.top_concepts import top_concepts

K = 20
concepts = torch.load(CONCEPTS_PT, map_location="cpu", weights_only=False)
top_idx  = top_concepts(TASK, codes_train, y_train, k=K)

print(f"Top-{K} concepts:")
for rank, idx in enumerate(top_idx, 1):
    print(f"  {rank:2d}. [{idx:4d}] {concepts[idx]}")

Top-20 concepts:
   1. [ 132] snowpack
   2. [   2] greenery
   3. [ 101] lake
   4. [ 193] esker
   5. [ 127] wilderness
   6. [ 178] moorland
   7. [ 192] drumlin
   8. [ 166] mudflat
   9. [ 161] embankment
  10. [  56] lagoon
  11. [ 117] topography
  12. [ 134] landslide
  13. [ 122] erosion
  14. [ 184] meander
  15. [  51] delta
  16. [   3] water
  17. [ 177] heathland
  18. [  91] jetty
  19. [   8] roadway
  20. [  31] wood


## 7. Concept manipulation (ablation / maximization)

In [10]:
from prediction.concept_manipulation import run_manipulation

rng      = np.random.default_rng(42)
rand_idx = rng.choice(codes_train.shape[1], len(top_idx), replace=False)

results = run_manipulation(TASK, probe_codes, codes_test, y_test, codes_train, top_idx, rand_idx)
for cond, val in results.items():
    print(f"  {cond:20s}  {val:.4f}  (delta {val - score_codes:+.4f})")

  original              0.4424  (delta +0.0000)
  zeroed                0.3818  (delta -0.0606)
  maximized             -0.7147  (delta -1.1570)
  random_zeroed         0.4463  (delta +0.0039)
